In [3]:
%cd /workspace/EBES

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from pathlib import Path
import optuna 
from ebes.pipeline.utils import optuna_df
from optuna.trial import TrialState

/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/workspace/EBES


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#%pip install plotly

In [4]:
def get_run(number, specify="best", rewrite=False):
    path = Path(f"log/{dataset}/{method}/optuna/{number}")
    print(pd.read_csv(path / "results.csv"))
    print((path / "params.txt").read_text())
    save_path = Path(f"configs/specify/{dataset}/{method}")
    save_path.mkdir(parents=True, exist_ok=True)
    save_path = (save_path / f"{specify}.yaml")
    if not rewrite:
        assert not save_path.exists()
    save_path.write_text((path / "params.txt").read_text())

def prepare_data(dataset, method):
    path = Path(f"log/{dataset}/{method}/optuna")
    df, study = optuna_df(path)
    col_to_drop = ["datetime_start", "datetime_complete", "system_attrs_fixed_params", "state", "value"]
    col_params = ["value", "duration"] + [col for col in df if "params_" in col]
    col_user = ["value", "duration"] + [col for col in df if "user" in col]
    df["duration"] = df["duration"].dt.total_seconds()
    return df, study, col_user, col_params

In [9]:
dataset = "age"
method = "coles64"
df, study, col_user, col_params = prepare_data(dataset, method)

print(df.shape, df[~df["value"].isna()].shape)
test_cols = [col for col in col_user if ("test" in col)]
df[~df["value"].isna()].sort_values("value", ascending=False).iloc[:10, ][["value"] + test_cols]

(123, 23) (87, 23)


/workspace/EBES/ebes/pipeline/utils.py:161: ExperimentalWarning: JournalStorage is experimental (supported from v3.1.0). The interface can change in the future.
  storage = JournalStorage(JournalFileStorage(f"{path}/study.log"))


,value
67,-21.962323
121,-22.357582
53,-22.509549
118,-23.044925
113,-23.111333
111,-23.369365
96,-24.469086
82,-25.144820
91,-25.221843
70,-26.053825


In [10]:
get_run(67, specify="best", rewrite=True)

     Unnamed: 0            0         mean  std
0    train_loss   -21.024106   -21.024106  0.0
1          loss   -21.962323   -21.962323  0.0
2  memory_after  2206.000000  2206.000000  0.0
optimizer:
  params:
    weight_decay: 1.4566614820256252e-08
    lr: 0.0013226951296751199
model:
  encoder:
    params:
      num_layers: 3
      dropout: 0.0011858822393245186
  preprocess:
    params:
      time_process: diff
      num_norm: false
      cat_emb_dim: 24
      num_emb_dim: 8
  aggregation:
    name: TakeLastHidden
unsupervised_loss:
  params:
    margin: 0.2018582996576839



In [38]:
failed = df[(df["state"] != "COMPLETE") | (df[col_user].isna().any(axis=1))][col_user].index
print(failed)
for fail in df[(df["state"] != "COMPLETE") | (df[col_user].isna().any(axis=1))][col_user].index:
    error_path = Path(f"/home/dev/24/es-bench/log/{dataset}/{method}/optuna/{fail}/ERROR.txt")
    if error_path.exists():
        error = error_path.read_text()
        print(fail, error.split("\n")[-2])
    else:
        print(df.loc[fail])

Index([ 0,  5, 10, 20, 22, 25, 26, 27, 29, 30, 32, 36, 37, 40, 43, 44, 46, 47,
       48, 51, 56, 57, 60, 61, 62, 63, 65, 66, 68, 69, 70],
      dtype='int64')
value                                                                                        NaN
datetime_start                                                        2026-02-20 13:32:36.835733
datetime_complete                                                     2026-02-20 13:46:13.181945
duration                                                                              816.346212
params_model.aggregation.name                                                    ValidHiddenMean
params_model.encoder.params.dropout                                                          0.0
params_model.encoder.params.num_layers                                                         1
params_model.preprocess.params.cat_emb_dim                                                    27
params_model.preprocess.params.num_emb_dim                      

In [39]:
optuna.visualization.plot_optimization_history(study)

### Params influence

In [ ]:
trials = study.trials
trials = [trial for trial in trials if trial.state == TrialState.COMPLETE]
plotted_trials = sorted(trials, key=lambda t: t.value)[:]
plotted_study = optuna.create_study()
for trial in plotted_trials:
    plotted_study.add_trial(trial)

[I 2026-02-08 14:14:01,944] A new study created in memory with name: no-name-bf32e8ce-34b0-42a4-b09f-effb0f848189


In [ ]:
target = None #lambda t: (t.user_attrs["memory_after_mean"])
target_name = "value"
fig = optuna.visualization.plot_param_importances(plotted_study, target=target, target_name=target_name)
print(fig._data[0]["x"][::-1])
print(fig._data[0]["y"][::-1])
take = 7
params = fig._data[0]["y"][-take:]
not_imp = list(set([col.replace("params_", "") for col in col_params]) - set(params) - {"duration", "value", "system_attrs_fixed_params"})
fig

[0.747853275855549, 0.1367710158504766, 0.07660412111367104, 0.02244571703129384, 0.006876204356437904, 0.004275955415268448, 0.003366077132665291, 0.0013274439885901082, 0.0002972588055507459, 0.00018293045049717313]
['unsupervised_loss.params.margin', 'optimizer.params.weight_decay', 'model.encoder.params.dropout', 'model.preprocess.params.num_emb_dim', 'model.preprocess.params.cat_emb_dim', 'model.preprocess.params.time_process', 'model.aggregation.name', 'model.encoder.params.num_layers', 'optimizer.params.lr', 'model.preprocess.params.num_norm']


In [ ]:
params

['model.aggregation.name',
 'model.preprocess.params.time_process',
 'model.preprocess.params.cat_emb_dim',
 'model.preprocess.params.num_emb_dim',
 'model.encoder.params.dropout',
 'optimizer.params.weight_decay',
 'unsupervised_loss.params.margin']

In [ ]:
# fig = optuna.visualization.plot_parallel_coordinate(plotted_study, target=target, target_name=target_name, params=['model.encoder.params.pooling', 'pretrain_model.encoder.params.pooling',])
# fig = optuna.visualization.plot_contour(study, target=target, target_name=target_name, params=params+not_imp)
fig = optuna.visualization.plot_slice(study, target=target, target_name=target_name)#, params=["model.encoder.params.num_layers"] )
# fig = optuna.visualization.plot_optimization_history(study, target=target, target_name=target_name, error_bar=False)
# targets = lambda t: (t.user_attrs["memory_after_mean"], t.user_attrs["val_metric_mean"])
# target_names = ["memory_after_mean", "val_metric_mean"]
# fig = optuna.visualization.plot_pareto_front(study, targets=targets, target_names=target_names)
fig